# Baseline Solution

## Описание

В этом Jupyter-ноутбуке представлено **базовое решение** задачи бинарной классификации: **Определение релевантности изображения для карточки товара.**

Данное решение:
- Простое и быстрое в реализации, так как не требует обучения модели с нуля.
- Использует предобученную модель [**jinaai/jina-clip-v2**](https://huggingface.co/jinaai/jina-clip-v2).
- Подходит для быстрого старта, улучшения, и поможет разобраться как правильно формировать файл решения.

## Imports and constants

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

warnings.filterwarnings("ignore")
tqdm.pandas()

Как уже было указано, в данном решении используется предобученная модель **CLIP (Contrastive Language–Image Pretraining)** от OpenAI —  а именно, её усовершенствованная версия от Jina AI:
[**jinaai/jina-clip-v2**](https://huggingface.co/jinaai/jina-clip-v2).

Решение основано на анализе семантической схожести пары **изображение — текст** (в нашем случае текст — это название товара и его описание).  

Модель Jina CLIP v2 расширяет оригинальную архитектуру CLIP, предлагая улучшенное понимание как на английском, так и на множестве других языков, включая русский, а также демонстрирует повышенную точность в задачах сопоставления изображений и текстов. Мы используем модель следующим образом:
1. Для каждой пары *изображение — название товара + его описание* вычисляется их **схожесть (similarity score)**. В качестве схожести используем **косинусную близость**.
2. Этот скор интерпретируется как **вероятность релевантности** изображения для данной карточки товара.

In [35]:
MODEL_NAME = "jinaai/jina-clip-v2"
DEVICE = "mps" if torch.mps.is_available() else "cpu"

# Замените пути на свои при необходимости
DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(f"Currently using {DEVICE}")

# MODEL_NAME = "jinaai/jina-clip-v2"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # Замените пути на свои при необходимости
# DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
# IMAGE_FOLDER = DATA_FOLDER / "images"

# print(f"Currently using {DEVICE}")

Currently using mps


## Model Initialization

Загружаем модель с huggingface. Будем использовать библиотеку [sentence_transformers](https://sbert.net) 

In [36]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)
model.to(DEVICE)

print(f"Model is currently on {model.device}")

KeyboardInterrupt: 

## Data Loading

Поскольку в данном решении мы не проводим обучение модели, необходимость в полной обучающей выборке отсутствует.  

Тем не менее, для оценки качества нашего подхода мы выделим небольшую **валидационную подвыборку** из обучающих данных. С её помошью:
- Проверим корректность реализации логики инференса.
- Оценим целевую метрику качества на размеченных данных.
- Получить приблизительное представление о том, насколько хорошо модель справляется с задачей определения релевантности.

В дальнейшем эта подвыборка будет обозначаться как `val_df`

In [37]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

val_df = train_df.head(500)

В качестве меры схожности, как было сказано ранее, применяется **косинусная близость (cosine similarity)** между векторными представлениями (эмбеддингами) изображения и текста.

Следующий код реализует: 
- Базовую предобработку текстов (конкатенацию названия и описания)
- Вычисление поэлементного косинусного сходства для соответствующих пар

Чем ближе значение косинусной близости к 1, тем выше предполагаемая релевантность изображения для данного текстового описания.

In [ ]:
def combine_texts(row):
    return f"Название товара: {row['name']}. Описание товара: {row['description']}"


batch_size = 32
similarities = []

image_paths = [os.path.join(IMAGE_FOLDER, f"{row_id}.jpg") for row_id in val_df["id"]]
texts = val_df.apply(combine_texts, axis=1).tolist()

for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
    text_batch = texts[i : i + batch_size]
    img_batch = image_paths[i : i + batch_size]

    text_embeds = model.encode(
        text_batch,
        normalize_embeddings=True,
        batch_size=len(text_batch),
        show_progress_bar=False,
    )
    image_embeds = model.encode(
        img_batch,
        normalize_embeddings=True,
        batch_size=len(img_batch),
        show_progress_bar=False,
    )

    text_embeds = torch.tensor(text_embeds).to(DEVICE)
    image_embeds = torch.tensor(image_embeds).to(DEVICE)

    cos_sims = util.cos_sim(text_embeds, image_embeds)
    diag_sims = torch.diag(cos_sims).cpu().numpy()

    similarities.extend(diag_sims)

val_df = val_df.copy()
val_df["similarity_score"] = similarities

Batches: 100%|██████████| 32/32 [01:33<00:00,  2.94s/it]


Теперь мы можем оценить качество полученных предсказаний с помощью метрики **ROC AUC**.

ROC AUC интерпретируется как вероятность того, что случайно выбранная положительная пара (релевантное изображение) получит от модели более высокий скор, чем случайно выбранная отрицательная пара (нерелевантное изображение).  
Значение:
- **ROC AUC = 0.5** — модель не лучше случайного угадывания.
- **ROC AUC > 0.5** — модель устойчиво различает классы.
- **ROC AUC < 0.5** — модель систематически "путает" классы: присваивает более высокие скоры нерелевантным изображениям, чем релевантным.

In [6]:
roc_auc_val = roc_auc_score(val_df["label"], val_df["similarity_score"])
print(f"ROC AUC на валидации:{roc_auc_val: .3f}")

ROC AUC на валидации: 0.454


Аналогично получим схожесть для тестовой выборки.

In [7]:
image_paths = [os.path.join(IMAGE_FOLDER, f"{row_id}.jpg") for row_id in test_df["id"]]
texts = test_df.apply(combine_texts, axis=1).tolist()

similarities = []

for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
    text_batch = texts[i : i + batch_size]
    img_batch = image_paths[i : i + batch_size]

    text_embeds = model.encode(
        text_batch,
        normalize_embeddings=True,
        batch_size=len(text_batch),
        show_progress_bar=False,
    )
    image_embeds = model.encode(
        img_batch,
        normalize_embeddings=True,
        batch_size=len(img_batch),
        show_progress_bar=False,
    )

    text_embeds = torch.tensor(text_embeds).to(DEVICE)
    image_embeds = torch.tensor(image_embeds).to(DEVICE)

    cos_sims = util.cos_sim(text_embeds, image_embeds)
    diag_sims = torch.diag(cos_sims).cpu().numpy()

    similarities.extend(diag_sims)

test_df["y_pred"] = similarities

Batches: 100%|██████████| 527/527 [32:42<00:00,  3.72s/it]


In [57]:
from torchvision import models, transforms
import torchvision.io as io
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [21]:
model_res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE)

In [68]:
tr = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [74]:
class ImbaDataset(Dataset):
    def __init__(self, root_dir, df, transform=None):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]
        self.transform = transform

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        else:
            transform_ = transforms.Compose([transforms.ToTensor()])
            image = transform_(image)
        
        return image

In [75]:
images_ds = ImbaDataset(IMAGE_FOLDER, test_df, tr)

In [76]:
images = DataLoader(images_ds, batch_size=batch_size, shuffle=False)

In [77]:
sureness = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure = model_res(batch.to(DEVICE))

    sure = image_sure.to('cpu').numpy()

    sureness.extend(sure)

100%|██████████| 527/527 [01:49<00:00,  4.81it/s]


## Нормализация сходства: приведение скоров к диапазону [0, 1]

Значения косинусной близости, возвращаемые моделью CLIP, лежат в диапазоне **от -1 до 1**, однако, для совместимости с **требованиями платформы соревнования**, предсказания **должны быть представлены в виде вероятностей**, то есть в диапазоне **[0, 1]**.

Это преобразование можно выполнить с помощью **сигмоидной функции**:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$
Применение сигмоиды не повлияет на метрику так как **сохраняет порядок** предсказаний — если одно изображение было оценено как более релевантное до преобразования, оно останется более релевантным после.

In [14]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


test_df["y_pred"] = sigmoid(test_df["y_pred"])

In [83]:
sureness = [[float(sigmoid(j)) for j in i] for i in sureness]

In [85]:
ans = []
for i in sureness:
    j = max(i)
    i[i.index(j)] = 0
    j -= max(i)
    ans.append(j)

In [90]:
print(similarities)

[np.float32(0.4687663), np.float32(0.43597877), np.float32(0.43113464), np.float32(0.45526892), np.float32(0.4082469), np.float32(0.34742543), np.float32(0.44022408), np.float32(0.39075908), np.float32(0.3606892), np.float32(0.40867752), np.float32(0.49454284), np.float32(0.3250508), np.float32(0.3831939), np.float32(0.3959343), np.float32(0.36095548), np.float32(0.412956), np.float32(0.39277667), np.float32(0.31712374), np.float32(0.38248196), np.float32(0.4099163), np.float32(0.39999723), np.float32(0.41157708), np.float32(0.42832392), np.float32(0.46172324), np.float32(0.3718904), np.float32(0.41090605), np.float32(0.47386834), np.float32(0.35612842), np.float32(0.46321988), np.float32(0.49312726), np.float32(0.31401014), np.float32(0.39403307), np.float32(0.40568367), np.float32(0.43266353), np.float32(0.4109147), np.float32(0.40667394), np.float32(0.35477012), np.float32(0.36169687), np.float32(0.37144947), np.float32(0.37638497), np.float32(0.4129061), np.float32(0.4508644), np.f

In [105]:
ans1 = []
for i in range(len(similarities)):
    ans1.append(sigmoid(similarities[i] + ans[i] * 1000))
test_df['y_pred'] = ans1

Сформируем csv файл решения. Полученный файл можно отправить на платформу.

In [106]:
test_df[["id", "y_pred"]].to_csv(DATA_FOLDER / "sample_submission.csv", index=None)

Если весь процесс прошёл удачно, то вы можете ожидать метрику на лидерборде $\approx 0.64$ ROC AUC